> **Note:** This notebook reuses patterns inspired by the Azure AI Gateway Labs (https://github.com/Azure-Samples/AI-Gateway).

# 🚀 Citadel Hosted Agent - Build, Deploy & Test

## Overview

This notebook builds a **Foundry Hosted Agent** using the Microsoft Agent Framework, deploys it to the Foundry project in the spoke resource group, and tests it end-to-end.

| Component | Detail |
|-----------|--------|
| **Agent Framework** | Microsoft Agent Framework (`agent-framework`) |
| **Protocol** | Responses API (hosted on port 8088) |
| **Model** | `gpt-4.1` via APIM BYO Gateway connection (`Hub-HR-ChatAgent-DEV-LLM/gpt-4.1`) |
| **Tools** | `get_current_time(timezone)`, `get_weather(location)` |
| **Telemetry** | OpenTelemetry → Application Insights (auto-injected by hosted agent platform) |
| **Container Build** | `az acr build` (cloud build — no local Docker required) |

## Prerequisites

- Citadel Governance Hub deployed (`azd up` completed)
- Spoke Foundry resources deployed (`workshop/scripts/deploy-spoke-foundry.ps1` or `.sh`)
- BYO Gateway connection `Hub-HR-ChatAgent-DEV-LLM` created (notebook 3)
- Python packages installed via `uv sync` in the `workshop/` directory
- Azure CLI logged in (`az login`)

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Read environment values from the `azd` environment and configure variables for this notebook.

In [ ]:
# Read environment values from azd
import subprocess, json

def azd_get_value(key):
    """Read a value from the current azd environment."""
    result = subprocess.run(
        ["azd", "env", "get-value", key],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to read {key}: {result.stderr.strip()}")
    return result.stdout.strip()

env_governance_hub_resource_group = azd_get_value("AZURE_RESOURCE_GROUP")
env_location = azd_get_value("AZURE_LOCATION")

env_foundry_subscription_id = azd_get_value("AZURE_SUBSCRIPTION_ID")
env_foundry_resource_group  = azd_get_value("SPOKE_RESOURCE_GROUP")
env_foundry_account_name    = azd_get_value("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
env_foundry_project_name    = azd_get_value("SPOKE_AI_FOUNDRY_PROJECT_NAME")
env_acr_name                = azd_get_value("SPOKE_ACR_NAME")
env_acr_login_server        = azd_get_value("SPOKE_ACR_LOGIN_SERVER")

print(f"Citadel Hub resource group: {env_governance_hub_resource_group}")
print(f"Location: {env_location}")
print(f"Foundry: {env_foundry_account_name} (Project: {env_foundry_project_name})")
print(f"Spoke RG: {env_foundry_resource_group}")
print(f"ACR: {env_acr_name} ({env_acr_login_server})")

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils

# ============================================================================
# 🔧 HOSTED AGENT CONFIGURATION
# To deploy a new version: bump hosted_agent_image_tag (e.g. "v1" → "v2"),
# then re-run from step 2 onward. The agent identity and RBAC persist
# across versions — only the image tag needs to change.
# ============================================================================
hosted_agent_name       = "citadel-time-weather-agent"
hosted_agent_image_tag  = "v1"
hosted_agent_model      = "Hub-HR-ChatAgent-DEV-LLM/gpt-4.1"  # BYO Gateway connection/model

# Derived values
hosted_agent_image      = f"{env_acr_login_server}/{hosted_agent_name}:{hosted_agent_image_tag}"
foundry_project_endpoint = f"https://{env_foundry_account_name}.services.ai.azure.com/api/projects/{env_foundry_project_name}"

utils.print_info(f"Agent name: {hosted_agent_name}")
utils.print_info(f"Image: {hosted_agent_image}")
utils.print_info(f"Model: {hosted_agent_model}")
utils.print_info(f"Foundry endpoint: {foundry_project_endpoint}")

<a id='1'></a>
### 1️⃣ Verify Azure CLI Login

Ensure you are logged in to Azure CLI and the correct subscription is active.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Generate Hosted Agent Source Files

This cell writes the agent application code into `workshop/hosted-agent/`. The agent is a simple **Time & Weather Assistant** with two local tools:

- **`get_current_time(timezone)`** — Returns the current date and time for a given timezone
- **`get_weather(location)`** — Returns simulated weather data (no external API needed)

The agent uses the **BYO Gateway connection** (`Hub-HR-ChatAgent-DEV-LLM/gpt-4.1`) to access the LLM through APIM — it does **not** deploy any models locally to the Foundry project.

**OpenTelemetry**: The hosted agent platform auto-injects `APPLICATIONINSIGHTS_CONNECTION_STRING` at runtime. The agent calls `client.configure_azure_monitor()` to enable distributed tracing, GenAI spans, and metrics in Application Insights.

> **Note:** The `RBAC` requirement for `az acr build` is **Contributor** on the ACR or the spoke resource group. The Foundry project managed identity needs **AcrPull** on the ACR (assigned by `deploy-spoke-foundry` script).

In [ ]:
import pathlib

agent_dir = pathlib.Path("hosted-agent")
agent_dir.mkdir(exist_ok=True)

# ---- main.py ----
main_py = r'''import os
from typing import Annotated
from datetime import datetime, timezone as tz

from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential


@tool(approval_mode="never_require")
def get_current_time(
    timezone: Annotated[str, Field(description="IANA timezone name, e.g. 'America/New_York', 'Europe/London', 'Asia/Tokyo'")]
) -> str:
    """Get the current date and time for a given timezone."""
    import zoneinfo
    try:
        zone = zoneinfo.ZoneInfo(timezone)
        now = datetime.now(zone)
        return f"Current time in {timezone}: {now.strftime('%Y-%m-%d %H:%M:%S %Z')}"
    except Exception as e:
        return f"Could not get time for timezone '{timezone}': {e}"


@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="City name, e.g. 'Seattle', 'London', 'Tokyo'")]
) -> str:
    """Get the current weather for a given location (simulated)."""
    import hashlib
    # Deterministic but varied simulated weather based on location + date
    seed = hashlib.md5(f"{location}{datetime.now(tz.utc).strftime('%Y-%m-%d')}".encode()).hexdigest()
    conditions = ["sunny", "partly cloudy", "cloudy", "rainy", "windy", "snowy"]
    condition = conditions[int(seed[:2], 16) % len(conditions)]
    temp_c = 5 + (int(seed[2:4], 16) % 30)
    humidity = 30 + (int(seed[4:6], 16) % 50)
    return f"Weather in {location}: {condition}, {temp_c} deg C, humidity {humidity}%"


def main():
    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=DefaultAzureCredential(),
    )

    agent = Agent(
        client=client,
        instructions=(
            "You are a helpful assistant that provides current time and weather information. "
            "Use the get_current_time tool for time queries and get_weather tool for weather queries. "
            "Be concise and friendly in your responses."
        ),
        tools=[get_current_time, get_weather],
        # History will be managed by the hosting infrastructure, thus there
        # is no need to store history by the service. Learn more at:
        # https://developers.openai.com/api/reference/resources/responses/methods/create
        default_options={"store": False},
    )

    server = ResponsesHostServer(agent)
    server.run()


if __name__ == "__main__":
    main()
'''
(agent_dir / "main.py").write_text(main_py.strip(), encoding="utf-8")
utils.print_ok("Created hosted-agent/main.py")

# ---- requirements.txt ----
requirements_txt = """\
agent-framework>=1.2.0
agent-framework-foundry-hosting>=1.0.0a260507
azure-identity>=1.25.0
"""
(agent_dir / "requirements.txt").write_text(requirements_txt, encoding="utf-8")
utils.print_ok("Created hosted-agent/requirements.txt")

# ---- Dockerfile ----
dockerfile = """\
FROM python:3.13-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .

EXPOSE 8088

CMD ["python", "main.py"]
"""
(agent_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")
utils.print_ok("Created hosted-agent/Dockerfile")

# ---- agent.yaml ----
agent_yaml = f"""\
kind: hosted
name: {hosted_agent_name}
protocols:
  - protocol: responses
    version: 1.0.0
resources:
  cpu: "1"
  memory: 2Gi
environment_variables:
  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
    value: {hosted_agent_model}
  - name: ENABLE_INSTRUMENTATION
    value: "true"
  - name: ENABLE_SENSITIVE_DATA
    value: "true"
  - name: OTEL_SERVICE_NAME
    value: {hosted_agent_name}
"""
(agent_dir / "agent.yaml").write_text(agent_yaml, encoding="utf-8")
utils.print_ok("Created hosted-agent/agent.yaml")

utils.print_info(f"\nAll agent source files written to: {agent_dir.resolve()}")

<a id='3'></a>
### 3️⃣ Build & Push Container Image

Uses `az acr build` to build the Docker image in the cloud and push it to ACR. **No local Docker installation required.**

> The signed-in user needs **Contributor** (or at minimum an ACR task-scheduling permission) on the ACR. If you deployed the spoke with `deploy-spoke-foundry`, you already have Owner on the spoke RG which inherits to the ACR.

In [ ]:
# NOTE: --no-logs is required because Azure CLI on Windows crashes streaming
# build logs due to a cp1252 encoding bug (pip output contains Unicode chars
# that colorama can't encode). The build runs server-side regardless.
acr_build_cmd = (
    f'az acr build'
    f' --registry {env_acr_name}'
    f' --resource-group {env_foundry_resource_group}'
    f' --image {hosted_agent_name}:{hosted_agent_image_tag}'
    f' --file hosted-agent/Dockerfile'
    f' --no-logs'
    f' hosted-agent/'
)

utils.print_info(f"Building image in ACR: {hosted_agent_image}")
utils.print_info("Build logs suppressed (Windows encoding bug). Polling for completion...")

output = utils.run(acr_build_cmd, "ACR build queued", "ACR build queue failed")

# Wait for the image to appear in ACR (build takes ~2-3 min)
import time as _t
for attempt in range(24):  # up to 4 minutes
    verify = utils.run(
        f'az acr repository show-tags --name {env_acr_name} --repository {hosted_agent_name} --output json',
        "", ""
    )
    if verify.success and verify.json_data and hosted_agent_image_tag in verify.json_data:
        utils.print_ok(f"Image verified in ACR: {hosted_agent_image}")
        break
    utils.print_info(f"  Waiting for image... ({(attempt+1)*10}s)")
    _t.sleep(10)
else:
    utils.print_error(f"Image '{hosted_agent_image_tag}' not found after 4 min. Check ACR build logs in the portal.")

<a id='4'></a>
### 4️⃣ Deploy Hosted Agent to Foundry

Creates a new hosted agent version in the Foundry project using the Python SDK. The agent is deployed with the container image from ACR and configured to use the BYO Gateway connection for LLM access.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential

utils.print_info(f"Connecting to Foundry endpoint: {foundry_project_endpoint}")

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=credential,
    allow_preview=True,
)
utils.print_ok(f"Foundry project client initialized for: {env_foundry_project_name}")

# Deploy the hosted agent
utils.print_info(f"Deploying hosted agent: {hosted_agent_name}")
utils.print_info(f"Image: {hosted_agent_image}")
utils.print_info(f"Model: {hosted_agent_model}")

agent = project_client.agents.create_version(
    agent_name=hosted_agent_name,
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(
                protocol=AgentProtocol.RESPONSES,
                version="1.0.0",
            )
        ],
        cpu="1",
        memory="2Gi",
        image=hosted_agent_image,
        environment_variables={
            "AZURE_AI_MODEL_DEPLOYMENT_NAME": hosted_agent_model,
            "ENABLE_INSTRUMENTATION": "true",
            "ENABLE_SENSITIVE_DATA": "true",
            "OTEL_SERVICE_NAME": hosted_agent_name,
        },
    ),
)

utils.print_ok(f"Agent version created: {agent.name} (version: {agent.version})")
utils.print_info(f"Status: {agent.status}")

<a id='5'></a>
### 5️⃣ Wait for Agent to Become Active

Poll the agent version status until it becomes `active`. This typically takes 2-5 minutes as the platform pulls the image and starts the container.

In [ ]:
max_wait_seconds = 600  # 10 minutes
poll_interval = 15

elapsed = 0
while elapsed < max_wait_seconds:
    version = project_client.agents.get_version(
        agent_name=agent.name,
        agent_version=agent.version,
    )
    status = version.status
    utils.print_info(f"[{elapsed}s] Agent status: {status}")

    if status == "active":
        utils.print_ok(f"Agent is active! (took ~{elapsed}s)")
        break
    elif status in ("failed", "error"):
        utils.print_error(f"Agent deployment failed with status: {status}")
        break

    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    utils.print_error(f"Timed out after {max_wait_seconds}s waiting for agent to become active.")

<a id='5a'></a>
### 5a. Assign RBAC to Agent Identity

Every hosted agent gets a dedicated Microsoft Entra ID (agent identity) at deploy time. This identity needs the **Foundry User** role on the Foundry account so it can access conversation storage and model endpoints.

> When deploying with `azd`, this role is assigned automatically. Since we're using the SDK, we assign it explicitly here.

In [ ]:
# Retrieve the agent identity assigned by the platform
agent_identity = version.instance_identity

if agent_identity is None:
    # Fallback: fetch from agent-level details
    agent_details = project_client.agents.get(agent_name=hosted_agent_name)
    agent_identity = agent_details.instance_identity

if agent_identity is None:
    utils.print_error("Could not resolve agent identity. Ensure the agent is in 'active' state.")
else:
    agent_principal_id = agent_identity.principal_id
    utils.print_ok(f"Agent identity principal ID: {agent_principal_id}")

    foundry_account_scope = (
        f"/subscriptions/{env_foundry_subscription_id}"
        f"/resourceGroups/{env_foundry_resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{env_foundry_account_name}"
    )

    # Check if Foundry User role is already assigned
    check = subprocess.run(
        ["az", "role", "assignment", "list",
         "--assignee-object-id", agent_principal_id,
         "--role", "Foundry User",
         "--scope", foundry_account_scope,
         "--query", "[0].id", "-o", "tsv"],
        capture_output=True, text=True, shell=True,
    )

    if check.stdout.strip():
        utils.print_ok("Foundry User role is already assigned.")
    else:
        utils.print_info(f"Assigning 'Foundry User' to agent identity on: {env_foundry_account_name}")
        result = subprocess.run(
            ["az", "role", "assignment", "create",
             "--assignee-object-id", agent_principal_id,
             "--assignee-principal-type", "ServicePrincipal",
             "--role", "Foundry User",
             "--scope", foundry_account_scope,
             "--only-show-errors", "-o", "none"],
            capture_output=True, text=True, shell=True,
        )
        if result.returncode == 0:
            utils.print_ok("Foundry User role assigned. Waiting 60s for RBAC propagation...")
            time.sleep(60)
            utils.print_ok("Done — RBAC should be active now.")
        else:
            utils.print_error(f"Failed to assign role: {result.stderr}")

<a id='6'></a>
### 6️⃣ Invoke the Hosted Agent

Use `project.get_openai_client(agent_name=...)` to invoke the hosted agent via the Responses API. This sends a request to the running agent container.

In [ ]:
import uuid

# Get an OpenAI client scoped to the hosted agent
openai_client = project_client.get_openai_client(agent_name=hosted_agent_name)

# Use a conversation_id so Foundry Traces groups requests together
conversation_id = str(uuid.uuid4())
utils.print_info(f"Conversation ID: {conversation_id}")

# Test 1: Time query
utils.print_info("Sending: What time is it in Tokyo?")
response = openai_client.responses.create(
    input="What time is it in Tokyo?",
    metadata={"conversation_id": conversation_id},
)

utils.print_ok(f"Agent response: {response.output_text}")
print(f"\nResponse ID: {response.id}")

# Test 2: Weather query
utils.print_info("\nSending: What's the weather in Seattle?")
response2 = openai_client.responses.create(
    input="What's the weather like in Seattle?",
    metadata={"conversation_id": conversation_id},
)

utils.print_ok(f"Agent response: {response2.output_text}")

<a id='7'></a>
### 7️⃣ Multi-Turn Conversation

Test a multi-turn conversation with the hosted agent using `previous_response_id` to maintain context across turns.

In [ ]:
conversation_messages = [
    "What time is it in New York and London right now?",
    "Now tell me the weather in both of those cities.",
    "Which city is warmer?",
]

previous_response_id = None
multi_turn_conversation_id = str(uuid.uuid4())
utils.print_info(f"Conversation ID: {multi_turn_conversation_id}")

for i, message in enumerate(conversation_messages, 1):
    utils.print_info(f"\n--- Turn {i} ---")
    utils.print_info(f"User: {message}")

    kwargs = {
        "input": message,
        "metadata": {"conversation_id": multi_turn_conversation_id},
    }
    if previous_response_id:
        kwargs["previous_response_id"] = previous_response_id

    response = openai_client.responses.create(**kwargs)
    previous_response_id = response.id

    utils.print_ok(f"Agent: {response.output_text}")

utils.print_info(f"\nConversation completed ({len(conversation_messages)} turns)")

<a id='8'></a>
### 8️⃣ Streaming Responses

Re-run the same queries with `stream=True` to demonstrate **streaming mode**. Tokens are printed as they arrive, significantly improving perceived latency (time-to-first-token) — the total inference time is similar, but the user sees output within seconds instead of waiting for the full response.

In [ ]:
# Streaming: single-turn queries
streaming_conversation_id = str(uuid.uuid4())
utils.print_info(f"Streaming Conversation ID: {streaming_conversation_id}")

# Test 1: Time query (streaming)
utils.print_info("Sending: What time is it in Tokyo?")
stream = openai_client.responses.create(
    input="What time is it in Tokyo?",
    metadata={"conversation_id": streaming_conversation_id},
    stream=True,
)
response = None
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        response = event.response
print()
utils.print_ok("Stream complete")
print(f"Response ID: {response.id}")

# Test 2: Weather query (streaming)
utils.print_info("\nSending: What's the weather in Seattle?")
stream2 = openai_client.responses.create(
    input="What's the weather like in Seattle?",
    metadata={"conversation_id": streaming_conversation_id},
    stream=True,
)
response2 = None
for event in stream2:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed":
        response2 = event.response
print()
utils.print_ok("Stream complete")

In [ ]:
# Streaming: multi-turn conversation
conversation_messages = [
    "What time is it in New York and London right now?",
    "Now tell me the weather in both of those cities.",
    "Which city is warmer?",
]

previous_response_id = None
streaming_multi_turn_id = str(uuid.uuid4())
utils.print_info(f"Streaming Conversation ID: {streaming_multi_turn_id}")

for i, message in enumerate(conversation_messages, 1):
    utils.print_info(f"\n--- Turn {i} ---")
    utils.print_info(f"User: {message}")

    kwargs = {
        "input": message,
        "metadata": {"conversation_id": streaming_multi_turn_id},
        "stream": True,
    }
    if previous_response_id:
        kwargs["previous_response_id"] = previous_response_id

    stream = openai_client.responses.create(**kwargs)
    response = None
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
        elif event.type == "response.completed":
            response = event.response
    print()
    previous_response_id = response.id

    utils.print_ok(f"Turn {i} complete")

utils.print_info(f"\nStreaming conversation completed ({len(conversation_messages)} turns)")

<a id='9'></a>
### 9️⃣ Cleanup

Delete the hosted agent entirely (all versions + agent identity). Set `cleanup_enabled = True` to remove the agent.

> **Note:** This removes the agent and its Entra ID identity from the Foundry project. The ACR image, RBAC role assignments, Foundry project, and other spoke resources are **not** affected. Orphaned RBAC assignments (for the deleted identity) are harmless and ignored by Azure.

In [ ]:
cleanup_enabled = False  # Set to True to enable cleanup of the agent after testing

if cleanup_enabled:
    try:
        result = project_client.agents.delete(agent_name=agent.name)
        utils.print_ok(f"Agent deleted: {agent.name} (deleted={result.get('deleted', True)})")
    except Exception as e:
        utils.print_warning(f"Failed to delete agent: {e}")
else:
    utils.print_info(f"Cleanup is disabled. Set cleanup_enabled = True to remove the agent.")
    utils.print_info(f"Agent retained: {agent.name} (version: {agent.version})")